In [0]:
%run ../config/config

In [0]:
import time
import sys
sys.path.insert(0, str(project_path))

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from utils.logger import MLOpsLogger
from utils.month_delta import month_delta

print("Imports completados")

In [0]:
logger = MLOpsLogger(
    name='06_quality_outputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={'mes_vta': mes_vta, 'environment': env, 'notebook': 'extract_persist'}
)

In [0]:
# =============================================================================
# EXTRACCIÓN DESDE LA LHCL TABLE + APLANAR STRUCT + PERSISTENCIA A DELTA
# =============================================================================
try:
    logger.log_stage_start('extract_persist', mes_vta=mes_vta, environment=env)
    start_time = time.time()

    spark = SparkSession.builder.getOrCreate()

    # Calcular rango de meses necesarios
    mes_limite_historico = month_delta(str(mes_vta), -meses_historicos_calidad)

    logger.info(f"Rango de extracción: {mes_limite_historico} a {mes_vta}")
    logger.info(f"Leyendo tabla LHCL: {output_table}")

    # ★ LECTURA ÚNICA de la LHCL table - filtrar todo lo necesario de una vez
    df_source = spark.read.format("delta").table(output_table)

    # ★ APLANAR struct colvariablesmodeloanalitico → columnas planas
    # Las columnas raíz fijas se combinan con todos los campos del struct
    struct_col = "colvariablesmodeloanalitico"
    if struct_col in df_source.columns:
        root_cols = [c for c in df_source.columns if c != struct_col]
        df_source = df_source.select(*root_cols, F.col(f"{struct_col}.*"))
        logger.info(f"✓ Struct '{struct_col}' aplanado: {len(df_source.columns)} columnas totales")
    else:
        logger.warning(f"Columna '{struct_col}' no encontrada - se procesa tabla tal como está")

    df_filtered = df_source.filter(
        (F.col("codmes") >= mes_limite_historico) & (F.col("codmes") <= mes_vta)
    )

    if DEBUG_MLOPS:
        logger.warning(f"MODO DEBUG: Limitando a {debug_rows} registros por mes")
        window = Window.partitionBy("codmes").orderBy("codclaveunicocli")
        df_filtered = df_filtered.withColumn("rn", F.row_number().over(window)) \
                                 .filter(F.col("rn") <= debug_rows).drop("rn")

    # ★ PERSISTIR A DELTA - esto rompe el lineage de la tabla fuente
    # Particionado por codmes para lecturas eficientes por mes
    persist_path = f"{adls_location_table}/quality_engine/{env}/{TABLE_OUTPUT}"

    logger.info(f"Persistiendo a Delta: {persist_path}")

    df_filtered.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("codmes") \
        .option("overwriteSchema", "true") \
        .save(persist_path)

    # Verificar lo que se escribió
    df_verify = spark.read.format("delta").load(persist_path)
    meses_persistidos = sorted([r.codmes for r in df_verify.select("codmes").distinct().collect()])
    total_persistido = df_verify.count()

    duration = time.time() - start_time

    logger.info(f"✅ Datos persistidos exitosamente")
    logger.info(f"  Path: {persist_path}")
    logger.info(f"  Meses: {meses_persistidos}")
    logger.info(f"  Total registros: {total_persistido:,}")
    logger.info(f"  Columnas: {len(df_verify.columns)}")
    logger.info(f"  Duración: {duration:.1f}s")

    logger.log_stage_end('extract_persist', duration=duration,
                        records_processed=total_persistido,
                        meses=len(meses_persistidos))

except Exception as e:
    logger.log_exception(operation='extract_persist', exception=e, should_raise=True)